# 04b · Label efficiency (C3)

**Purpose.** On GasWindTunnel, train with 10%, 25%, 50% and 100% of each training fold and plot macro F1 against the fraction of labels, one line per representation.

**Inputs**
- GasWindTunnel `X_signals.npy` (600, 72, 1500) + `meta.csv`; training fold of 480 → 48 / 120 / 240 / 480 samples

**Outputs**
- `results/04b_label_efficiency.csv`
- `report/figures/` label efficiency plot

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import load_baseline_level, load_processed, precomputed, run_cv, save_results
from src.features.encoder import encoder_builder
from src.features.handcrafted import handcrafted_features
from src.features.raw import raw_features

DATASET = "gwt"
TASK = "multi"
FRACTIONS = [0.10, 0.25, 0.50]     # 100% is read from the main results (01a, 02b, 03a)
NAME = "04b_label_efficiency"
FIG = ROOT / "report/figures"

X, meta, info = load_processed(DATASET)                            # (600, 72, 1500)
levels = load_baseline_level(DATASET)                              # (600, 72)
F_raw = raw_features(X, 100)                                       # (600, 7200)
F_hand = handcrafted_features(X, levels, info["n_base"], info["exposure_end"])   # (600, 864)

## C3: train on 10%, 25%, 50% of each training fold
The subsample is stratified and drawn **from the training fold only**; the test fold is unchanged.
The encoder (R3) is retrained on the smaller training set.

In [ ]:
runs = [("R1_raw", F_raw, precomputed), ("R2_handcrafted", F_hand, precomputed),
        ("R3_encoder", X, encoder_builder(n_out=32))]
parts = []
for frac in FRACTIONS:
    for rep, data, builder in runs:
        s, _ = run_cv(data, meta, DATASET, TASK, rep, builder, train_fraction=frac, verbose=False)
        parts.append(s)
        print(f"{frac:.2f} {rep}: " + "  ".join(f"{h} {v:.3f}" for h, v in s.groupby("head")["macro_f1"].mean().items()))
scores = pd.concat(parts, ignore_index=True)
save_results(scores, None, NAME)

In [ ]:
full = pd.concat([pd.read_csv(ROOT / f"results/{n}.csv") for n in ["01a_gwt_raw", "02b_gwt_handcrafted", "03a_gwt_encoder"]])
full = full[full["task"] == TASK]
allf = pd.concat([scores, full], ignore_index=True)
curve = allf.groupby(["head", "rep", "train_fraction"])["macro_f1"].agg(["mean", "std"]).reset_index()

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, head in zip(axes, ["logreg", "svm", "rf", "mlp"]):
    for rep in ["R1_raw", "R2_handcrafted", "R3_encoder"]:
        c = curve[(curve["head"] == head) & (curve["rep"] == rep)]
        ax.errorbar(c["train_fraction"] * 100, c["mean"], yerr=c["std"], marker="o", capsize=3, label=rep)
    ax.set(xscale="log", xticks=[10, 25, 50, 100], xticklabels=["10", "25", "50", "100"],
           xlabel="% of training fold", title=head)
axes[0].set_ylabel("macro F1 (GasWindTunnel, multiclass)")
axes[0].legend()
plt.tight_layout()
plt.savefig(FIG / "04b_label_efficiency.png", dpi=150)
plt.show()
curve.pivot_table(index=["rep", "head"], columns="train_fraction", values="mean").round(3)